# SQL assignment on data given in word 

### Word document: https://celebaltech.sharepoint.com/:w:/s/Celebal-LMS/IQDQ_Co-E08_RZo3eKklxGR-AQ09RkM5df72wSBZjaL9LYc?e=04TLA8

#### Create a database

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("shopease.db")
conn.execute("PRAGMA foreign_keys = ON")

#### Create customers table

In [3]:
conn.execute("""
CREATE TABLE customers (
    customer_id INT PRIMARY KEY,
    first_name VARCHAR(50) NOT NULL,
    last_name VARCHAR(50) NOT NULL,
    email VARCHAR(100) UNIQUE NOT NULL,
    city VARCHAR(50) NOT NULL,
    state VARCHAR(50) NOT NULL,
    join_date DATE NOT NULL,
    is_premium BOOLEAN DEFAULT FALSE
)
""")

#### Create products table

In [4]:
conn.execute("""
CREATE TABLE products (
    product_id INT PRIMARY KEY,
    product_name VARCHAR(100) NOT NULL,
    category VARCHAR(50) NOT NULL,
    brand VARCHAR(50) NOT NULL,
    unit_price DECIMAL(10,2) NOT NULL CHECK (unit_price > 0),
    stock_qty INT NOT NULL DEFAULT 0 CHECK (stock_qty >= 0)
)
""")

#### Create orders table

In [5]:
conn.execute("""
CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT NOT NULL,
    order_date DATE NOT NULL,
    status VARCHAR(20) NOT NULL DEFAULT 'Pending'
           CHECK (status IN ('Pending','Shipped','Delivered','Cancelled')),
    total_amount DECIMAL(12,2) NOT NULL CHECK (total_amount >= 0),

    FOREIGN KEY (customer_id)
    REFERENCES customers(customer_id)
)
""")

#### Create order_items table

In [6]:
conn.execute("""
CREATE TABLE order_items (
    item_id INT PRIMARY KEY,
    order_id INT NOT NULL,
    product_id INT NOT NULL,
    quantity INT NOT NULL CHECK (quantity > 0),
    unit_price DECIMAL(10,2) NOT NULL CHECK (unit_price > 0),
    discount_pct DECIMAL(5,2) DEFAULT 0 CHECK (discount_pct BETWEEN 0 AND 100),

    FOREIGN KEY (order_id)
    REFERENCES orders(order_id),

    FOREIGN KEY (product_id)
    REFERENCES products(product_id)
)
""")

In [7]:
#### Creating indexes

In [8]:
conn.execute("""
CREATE INDEX idx_customers_city
ON customers(city)
""")

conn.execute("""
CREATE INDEX idx_customers_state
ON customers(state)
""")

conn.execute("""
CREATE INDEX idx_products_category
ON products(category)
""")

conn.execute("""
CREATE INDEX idx_orders_date
ON orders(order_date)
""")

conn.execute("""
CREATE INDEX idx_orders_status
ON orders(status)
""")

### Data Loading — INSERT Statements 

#### Insert : customers 

In [9]:
conn.executescript("""
INSERT INTO customers VALUES 
(101, 'Aarav',  'Sharma', 'aarav.s@email.com',  'Mumbai',    'Maharashtra', '2024-01-15', TRUE), 
(102, 'Priya',  'Patel',  'priya.p@email.com',  'Ahmedabad', 'Gujarat',     '2024-02-20', FALSE), 
(103, 'Rohan',  'Gupta',  'rohan.g@email.com',  'Delhi',     'Delhi',       '2024-03-10', TRUE), 
(104, 'Sneha',  'Reddy',  'sneha.r@email.com',  'Hyderabad', 'Telangana',   '2024-04-05', FALSE), 
(105, 'Vikram', 'Singh',  'vikram.s@email.com', 'Jaipur',    'Rajasthan',   '2024-05-12', TRUE), 
(106, 'Ananya', 'Iyer',   'ananya.i@email.com', 'Chennai',   'Tamil Nadu',  '2024-06-18', FALSE), 
(107, 'Karan',  'Mehta',  'karan.m@email.com',  'Pune',      'Maharashtra', '2024-07-22', TRUE), 
(108, 'Divya',  'Nair',   'divya.n@email.com',  'Kochi',     'Kerala',      '2024-08-30', FALSE); 
""")

In [10]:
pd.read_sql("""
SELECT *
FROM customers
""", conn)

,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


#### Insert : products

In [11]:
conn.executescript("""
INSERT INTO products VALUES
(201, 'Wireless Earbuds',     'Electronics', 'BoAt',         1499.00, 250),
(202, 'Cotton T-Shirt',       'Clothing',    'Levis',         799.00, 500),
(203, 'Smart Watch',          'Electronics', 'Noise',        2999.00, 150),
(204, 'Running Shoes',        'Clothing',    'Nike',         4599.00, 120),
(205, 'Bluetooth Speaker',    'Electronics', 'JBL',          3499.00, 200),
(206, 'Bedsheet Set',         'Home',        'Spaces',       1299.00, 300),
(207, 'Laptop Stand',         'Electronics', 'AmazonBasics',  899.00, 180),
(208, 'Cushion Covers (Set)', 'Home',        'HomeCenter',    599.00, 400);
""")

In [12]:
pd.read_sql("""
SELECT *
FROM products
""", conn)

,product_id,product_name,category,brand,unit_price,stock_qty
0,201,Wireless Earbuds,Electronics,BoAt,1499,250
1,202,Cotton T-Shirt,Clothing,Levis,799,500
2,203,Smart Watch,Electronics,Noise,2999,150
3,204,Running Shoes,Clothing,Nike,4599,120
4,205,Bluetooth Speaker,Electronics,JBL,3499,200
5,206,Bedsheet Set,Home,Spaces,1299,300
6,207,Laptop Stand,Electronics,AmazonBasics,899,180
7,208,Cushion Covers (Set),Home,HomeCenter,599,400


#### Insert : orders

In [13]:
conn.executescript("""
INSERT INTO orders VALUES
(1001, 101, '2024-08-01', 'Delivered', 4498.00),
(1002, 102, '2024-08-03', 'Delivered', 799.00),
(1003, 103, '2024-08-05', 'Shipped', 7498.00),
(1004, 101, '2024-08-10', 'Delivered', 3499.00),
(1005, 104, '2024-08-12', 'Cancelled', 2999.00),
(1006, 105, '2024-08-15', 'Delivered', 5898.00),
(1007, 106, '2024-08-18', 'Pending', 1299.00),
(1008, 103, '2024-08-20', 'Delivered', 899.00),
(1009, 107, '2024-08-25', 'Shipped', 6098.00),
(1010, 108, '2024-08-28', 'Delivered', 1598.00);
""")

In [14]:
pd.read_sql("""
SELECT *
FROM orders
""", conn)

,order_id,customer_id,order_date,status,total_amount
0,1001,101,2024-08-01,Delivered,4498
1,1002,102,2024-08-03,Delivered,799
2,1003,103,2024-08-05,Shipped,7498
3,1004,101,2024-08-10,Delivered,3499
4,1005,104,2024-08-12,Cancelled,2999
5,1006,105,2024-08-15,Delivered,5898
6,1007,106,2024-08-18,Pending,1299
7,1008,103,2024-08-20,Delivered,899
8,1009,107,2024-08-25,Shipped,6098
9,1010,108,2024-08-28,Delivered,1598


#### Insert : order_items

In [15]:
conn.executescript("""
INSERT INTO order_items VALUES
(5001,1001,201,2,1499.00,0),
(5002,1001,207,1,899.00,10),
(5003,1002,202,1,799.00,0),
(5004,1003,203,1,2999.00,0),
(5005,1003,204,1,4599.00,5),
(5006,1004,205,1,3499.00,0),
(5007,1005,203,1,2999.00,0),
(5008,1006,201,1,1499.00,10),
(5009,1006,204,1,4599.00,5),
(5010,1007,206,1,1299.00,0),
(5011,1008,207,1,899.00,0),
(5012,1009,205,1,3499.00,0),
(5013,1009,208,2,599.00,15),
(5014,1010,206,1,1299.00,0),
(5015,1010,208,1,599.00,0);
""")

In [16]:
pd.read_sql("""
SELECT *
FROM order_items
""",conn)

,item_id,order_id,product_id,quantity,unit_price,discount_pct
0,5001,1001,201,2,1499,0
1,5002,1001,207,1,899,10
2,5003,1002,202,1,799,0
3,5004,1003,203,1,2999,0
4,5005,1003,204,1,4599,5
5,5006,1004,205,1,3499,0
6,5007,1005,203,1,2999,0
7,5008,1006,201,1,1499,10
8,5009,1006,204,1,4599,5
9,5010,1007,206,1,1299,0


# Section A — SQL Basics (SELECT, Constraints, Primary Keys) 

### Q1. Write a query to display all columns and rows from the customer's table.

In [17]:
query = """
SELECT *
FROM customers
"""

pd.read_sql(query, conn)

,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


### Q2. Retrieve only the first_name, last_name, and city of all customers.

In [18]:
query = """
SELECT
    first_name,
    last_name,
    city
FROM customers
"""

pd.read_sql(query, conn)

,first_name,last_name,city
0,Aarav,Sharma,Mumbai
1,Priya,Patel,Ahmedabad
2,Rohan,Gupta,Delhi
3,Sneha,Reddy,Hyderabad
4,Vikram,Singh,Jaipur
5,Ananya,Iyer,Chennai
6,Karan,Mehta,Pune
7,Divya,Nair,Kochi


### Q3. List all unique categories available in the products table. 

In [19]:
query = """
SELECT DISTINCT category
FROM products
"""

pd.read_sql(query, conn)

,category
0,Clothing
1,Electronics
2,Home


### Q4. Identify the Primary Key of each table in the schema. Explain why a Primary Key must be unique and NOT NULL. 

customers    -> customer_id

products     -> product_id

orders       -> order_id

order_items  -> item_id

A primary key must be UNIQUE so that every single row can be distinctly identified without any ambiguity, and NOT NULL because a missing or unknown value cannot act as a valid, permanent identifier.

### Q5. What constraints are applied to the email column in the customers table? What would happen if you tried to insert a duplicate email? 

email has:
- UNIQUE
- NOT NULL

In [20]:
#conn.execute("""
#INSERT INTO customers
#VALUES (
#109,
#'Aarav',
#'Shetty',
#'aarav.s@email.com',
#'Mumbai',
#'Maharashtra',
#'2024-09-01',
#TRUE
#)
#""")

IntegrityError: UNIQUE constraint failed: customers.email

Error=> 
IntegrityError: UNIQUE constraint failed: customers.email

### Q6. Try inserting a product with unit_price = -50. What happens and which constraint prevents it? Write both the INSERT statement and explain the error. 

In [57]:
#conn.execute("""
#INSERT INTO products
#VALUES (
#209,
#'Test Product',
#'Electronics',
#'TestBrand',
#-50,
#10
#)
#""")

SyntaxError: unterminated triple-quoted string literal (detected at line 11) (2981988615.py, line 11)

IntegrityError: CHECK constraint failed: unit_price > 0Constraint responsible: CHECK (unit_price > 0) prevents negative product prices

# Section B — Filtering & Optimization (WHERE, Indexes) 

### Q7. Retrieve all orders with status = 'Delivered'. 

In [22]:
query = """
SELECT *
FROM orders
WHERE status = 'Delivered'
"""

pd.read_sql(query, conn)

,order_id,customer_id,order_date,status,total_amount
0,1001,101,2024-08-01,Delivered,4498
1,1002,102,2024-08-03,Delivered,799
2,1004,101,2024-08-10,Delivered,3499
3,1006,105,2024-08-15,Delivered,5898
4,1008,103,2024-08-20,Delivered,899
5,1010,108,2024-08-28,Delivered,1598


Displays all orders that have been successfully delivered to customers.

### Q8. Find all products in the 'Electronics' category with a unit_price greater than ₹2000.

In [23]:
query = """
SELECT *
FROM products
WHERE category = 'Electronics'
AND unit_price > 2000
"""

pd.read_sql(query, conn)

,product_id,product_name,category,brand,unit_price,stock_qty
0,203,Smart Watch,Electronics,Noise,2999,150
1,205,Bluetooth Speaker,Electronics,JBL,3499,200


### Q9. List all customers who joined in the year 2024 and belong to the state 'Maharashtra'. 

In [24]:
query = """
SELECT *
FROM customers
WHERE state = 'Maharashtra'
AND join_date BETWEEN '2024-01-01' AND '2024-12-31'
"""

pd.read_sql(query, conn)

,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1


### Q10. Find all orders placed between '2024-08-10' and '2024-08-25' (inclusive) that are NOT cancelled

In [25]:
query = """
SELECT *
FROM orders
WHERE order_date BETWEEN '2024-08-10' AND '2024-08-25'
AND status <> 'Cancelled'
"""

pd.read_sql(query, conn)

,order_id,customer_id,order_date,status,total_amount
0,1004,101,2024-08-10,Delivered,3499
1,1006,105,2024-08-15,Delivered,5898
2,1007,106,2024-08-18,Pending,1299
3,1008,103,2024-08-20,Delivered,899
4,1009,107,2024-08-25,Shipped,6098


### Q11. Explain what the index idx_orders_date does. How would it improve performance?

The index idx_orders_date is created on the order_date column. It helps the database locate matching dates faster instead of scanning every row in the table. This improves the performance of date-based filtering queries.

In [26]:
query = """
SELECT *
FROM orders
WHERE order_date BETWEEN '2024-08-01' AND '2024-08-31'
"""

pd.read_sql(query, conn)

,order_id,customer_id,order_date,status,total_amount
0,1001,101,2024-08-01,Delivered,4498
1,1002,102,2024-08-03,Delivered,799
2,1003,103,2024-08-05,Shipped,7498
3,1004,101,2024-08-10,Delivered,3499
4,1005,104,2024-08-12,Cancelled,2999
5,1006,105,2024-08-15,Delivered,5898
6,1007,106,2024-08-18,Pending,1299
7,1008,103,2024-08-20,Delivered,899
8,1009,107,2024-08-25,Shipped,6098
9,1010,108,2024-08-28,Delivered,1598


### Q12. If you run: SELECT * FROM customers WHERE YEAR(join_date) = 2024; — would the index on join_date be used? Explain why or why not, and rewrite the query to be index-friendly (SARGable). 

SELECT *
FROM customers
WHERE YEAR(join_date) = 2024;

The above query is not index friendly as the index is generally not used efficiently because the YEAR() function is applied to every value in the join_date column before comparison.

In [27]:
# Index-Friendly (SARGable) Version:
query = """
SELECT *
FROM customers
WHERE join_date BETWEEN '2024-01-01' AND '2024-12-31'
"""

pd.read_sql(query, conn)

,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


This query is SARGable because it directly filters on the indexed column using a date range, allowing the database to use the index efficiently.

# Section C — Aggregation (GROUP BY, SUM, COUNT, AVG, MIN, MAX)

### Q13. Count the total number of orders in the orders table. 

In [28]:
query = """
SELECT COUNT(*) AS total_orders
FROM orders
"""

pd.read_sql(query, conn)

,total_orders
0,10


### Q14. Find the total revenue (SUM of total_amount) from all 'Delivered' orders.

In [29]:
query = """
SELECT SUM(total_amount) AS total_revenue
FROM orders
WHERE status = 'Delivered'
"""

pd.read_sql(query, conn)

,total_revenue
0,17191


### Q15. Calculate the average unit_price of products in each category.

In [30]:
query = """
SELECT
    category,
    AVG(unit_price) AS avg_price
FROM products
GROUP BY category
"""

pd.read_sql(query, conn)

,category,avg_price
0,Clothing,2699.0
1,Electronics,2224.0
2,Home,949.0


### Q16. For each order status, find the count of orders and the total revenue. Sort the result by total revenue in descending order. 

In [31]:
query = """
SELECT
    status,
    COUNT(*) AS total_orders,
    SUM(total_amount) AS total_revenue
FROM orders
GROUP BY status
ORDER BY total_revenue DESC
"""

pd.read_sql(query, conn)

,status,total_orders,total_revenue
0,Delivered,6,17191
1,Shipped,2,13596
2,Cancelled,1,2999
3,Pending,1,1299


### Q17. Find the most expensive (MAX) and cheapest (MIN) product in each category.

In [32]:
query = """
SELECT
    category,
    MAX(unit_price) AS max_price,
    MIN(unit_price) AS min_price
FROM products
GROUP BY category
"""

pd.read_sql(query, conn)

,category,max_price,min_price
0,Clothing,4599,799
1,Electronics,3499,899
2,Home,1299,599


### Q18. List all product categories where the average unit_price is greater than ₹2000. (HAVING clause)

In [33]:
query = """
SELECT
    category,
    AVG(unit_price) AS avg_price
FROM products
GROUP BY category
HAVING AVG(unit_price) > 2000
"""

pd.read_sql(query, conn)

,category,avg_price
0,Clothing,2699.0
1,Electronics,2224.0


# Section D — Joins & Relationships 

### Q19. Write an INNER JOIN query to display each order along with the customer's first_name and last_name.

In [34]:
query = """
SELECT
    o.order_id,
    o.order_date,
    c.first_name,
    c.last_name,
    o.total_amount
FROM orders o
INNER JOIN customers c
ON o.customer_id = c.customer_id
"""

pd.read_sql(query, conn)

,order_id,order_date,first_name,last_name,total_amount
0,1001,2024-08-01,Aarav,Sharma,4498
1,1002,2024-08-03,Priya,Patel,799
2,1003,2024-08-05,Rohan,Gupta,7498
3,1004,2024-08-10,Aarav,Sharma,3499
4,1005,2024-08-12,Sneha,Reddy,2999
5,1006,2024-08-15,Vikram,Singh,5898
6,1007,2024-08-18,Ananya,Iyer,1299
7,1008,2024-08-20,Rohan,Gupta,899
8,1009,2024-08-25,Karan,Mehta,6098
9,1010,2024-08-28,Divya,Nair,1598


### Q20. Using a LEFT JOIN, list ALL customers and their orders (if any). Customers with no orders should still appear with NULL values for order columns. 

Insert a customer with no order, no understand output clearly

In [38]:
conn.execute("""
INSERT INTO customers
VALUES (
109,
'Rahul',
'Verma',
'rahul@email.com',
'Bhopal',
'Madhya Pradesh',
'2024-09-01',
FALSE
)
""")

conn.commit()

In [39]:
query = """
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    o.order_id,
    o.order_date,
    o.total_amount
FROM customers c
LEFT JOIN orders o
ON c.customer_id = o.customer_id
"""

pd.read_sql(query, conn)

,customer_id,first_name,last_name,order_id,order_date,total_amount
0,101,Aarav,Sharma,1001.0,2024-08-01,4498.0
1,101,Aarav,Sharma,1004.0,2024-08-10,3499.0
2,102,Priya,Patel,1002.0,2024-08-03,799.0
3,103,Rohan,Gupta,1003.0,2024-08-05,7498.0
4,103,Rohan,Gupta,1008.0,2024-08-20,899.0
5,104,Sneha,Reddy,1005.0,2024-08-12,2999.0
6,105,Vikram,Singh,1006.0,2024-08-15,5898.0
7,106,Ananya,Iyer,1007.0,2024-08-18,1299.0
8,107,Karan,Mehta,1009.0,2024-08-25,6098.0
9,108,Divya,Nair,1010.0,2024-08-28,1598.0


The left join returns all customers, including Rahul who has no orders.

### Q21. Write a query using JOINs across three tables (orders → order_items → products) to show: order_id, product_name, quantity, unit_price, and discount_pct for each order item. 

In [36]:
query = """
SELECT
    o.order_id,
    p.product_name,
    oi.quantity,
    oi.unit_price,
    oi.discount_pct
FROM orders o
INNER JOIN order_items oi
ON o.order_id = oi.order_id
INNER JOIN products p
ON oi.product_id = p.product_id
"""

pd.read_sql(query, conn)

,order_id,product_name,quantity,unit_price,discount_pct
0,1001,Wireless Earbuds,2,1499,0
1,1001,Laptop Stand,1,899,10
2,1002,Cotton T-Shirt,1,799,0
3,1003,Smart Watch,1,2999,0
4,1003,Running Shoes,1,4599,5
5,1004,Bluetooth Speaker,1,3499,0
6,1005,Smart Watch,1,2999,0
7,1006,Wireless Earbuds,1,1499,10
8,1006,Running Shoes,1,4599,5
9,1007,Bedsheet Set,1,1299,0


### Q22. Explain the difference between LEFT JOIN and RIGHT JOIN with an example from this schema. When would you use a FULL OUTER JOIN?

A LEFT JOIN returns all records from the left table and matching records from the right table. If no match exists, the columns from the right table contain NULL.

In [40]:
query = """
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    o.order_id,
    o.order_date,
    o.total_amount
FROM customers c
LEFT JOIN orders o
ON c.customer_id = o.customer_id
"""

pd.read_sql(query, conn)

,customer_id,first_name,last_name,order_id,order_date,total_amount
0,101,Aarav,Sharma,1001.0,2024-08-01,4498.0
1,101,Aarav,Sharma,1004.0,2024-08-10,3499.0
2,102,Priya,Patel,1002.0,2024-08-03,799.0
3,103,Rohan,Gupta,1003.0,2024-08-05,7498.0
4,103,Rohan,Gupta,1008.0,2024-08-20,899.0
5,104,Sneha,Reddy,1005.0,2024-08-12,2999.0
6,105,Vikram,Singh,1006.0,2024-08-15,5898.0
7,106,Ananya,Iyer,1007.0,2024-08-18,1299.0
8,107,Karan,Mehta,1009.0,2024-08-25,6098.0
9,108,Divya,Nair,1010.0,2024-08-28,1598.0


In our example, we added customer Rahul Verma (customer_id = 109) who has not placed any orders. The query still displays Rahul, but the order-related columns contain NULL.

A RIGHT JOIN returns all records from the right table and matching records from the left table. If no match exists, the columns from the left table contain NULL.
Example=> 
- SELECT
            c.customer_id,
            c.first_name,
            o.order_id
            FROM customers c
            RIGHT JOIN orders o
            ON c.customer_id = o.customer_id;

If an order existed without a matching customer, the order would still be displayed and the customer columns would contain NULL.

Difference
- LEFT JOIN keeps all rows from the left table (customers).
- RIGHT JOIN keeps all rows from the right table (orders).
- Missing matches appear as NULL.

When would you use a FULL OUTER JOIN?

A FULL OUTER JOIN returns:
- All matching records from both tables.
- Unmatched rows from the left table.
- Unmatched rows from the right table.

It is useful when we want to identify records that do not have corresponding matches in either table.

Example=>

To find:
Customers who have not placed any orders.
Orders that do not have a valid customer.

A FULL OUTER JOIN would return both types of unmatched records in a single result set.

### Q23. Identify all Foreign Key relationships in the schema. Explain what would happen if you tried to insert an order with customer_id = 999 (which doesn't exist in customers).

Foreign keys: 
- orders.customer_id → customers.customer_id
- order_items.order_id → orders.order_id
- order_items.product_id → products.product_id


What happens if customer_id = 999 is inserted into orders?

In [37]:
#conn.execute("""
#INSERT INTO orders
#VALUES (
#1011,
#999,
#'2024-09-01',
#'Pending',
#1000
#)
#""")

IntegrityError: FOREIGN KEY constraint failed

Error=> IntegrityError: FOREIGN KEY constraint failed

# Section E — Advanced Concepts (CASE, ACID, Transactions)

### Q24. Write a query using CASE to classify products into price tiers: 
 

- 'Budget'    → unit_price < 1000 
- 'Mid-Range' → unit_price BETWEEN 1000 AND 3000 
- 'Premium'   → unit_price > 3000 

Display: product_name, unit_price, price_tier.

In [47]:
query = """
SELECT
    product_name,
    unit_price,
    CASE
        WHEN unit_price < 1000 THEN 'Budget'
        WHEN unit_price BETWEEN 1000 AND 3000 THEN 'Mid-Range'
        ELSE 'Premium'
    END AS price_tier
FROM products
"""

pd.read_sql(query, conn)

,product_name,unit_price,price_tier
0,Wireless Earbuds,1499,Mid-Range
1,Cotton T-Shirt,799,Budget
2,Smart Watch,2999,Mid-Range
3,Running Shoes,4599,Premium
4,Bluetooth Speaker,3499,Premium
5,Bedsheet Set,1299,Mid-Range
6,Laptop Stand,899,Budget
7,Cushion Covers (Set),599,Budget


### Q25. Using a CASE statement inside an aggregate function, count how many orders are 'Delivered' vs 'Not Delivered' (all other statuses). Display the result in a single row. 

In [51]:
query = """
SELECT
    COUNT(CASE WHEN status = 'Delivered' THEN 1 END) AS delivered_orders,
    COUNT(CASE WHEN status <> 'Delivered' THEN 1 END) AS not_delivered_orders
FROM orders
"""

pd.read_sql(query, conn)

,delivered_orders,not_delivered_orders
0,6,4


### Q26. Explain each letter of ACID: 

- A – Atomicity 
- C – Consistency 
- I – Isolation 
- D – Durability 


Give a real-world example (e.g., bank transfer) showing why each property is important. 

Atomicity means a transaction is completed fully or not executed at all.

Example: During a transfer of ₹1000 from Account A to B, if the amount is deducted from A but a system failure occurs before adding it to B, the entire transaction is rolled back. This prevents inconsistent updates.

Consistency ensures that data remains valid before and after a transaction.

Example: An attempted withdrawal that forces an account balance below 0 gets rejected to maintain the required valid state.

Isolation ensures that multiple transactions running at the same time do not interfere with each other.

Example: If two users try to withdraw money from the same account simultaneously, isolation prevents incorrect balance calculations and ensures each transaction behaves as if it were executed separately.

Durability ensures that once a transaction is successfully committed, it is permanently saved.

Example: After a bank transfer is completed and confirmed, the updated balances remain saved even if there is a power failure or system crash immediately afterward.

ACID properties are important because they ensure transactions are reliable, accurate, and safe even when multiple users access the database or unexpected failures occur.


### Q27. Write a SQL transaction that does the following atomically: 


  1. Insert a new order (order_id=1011, customer_id=102, today's date, 'Pending', 1598.00) 
  2. Insert two order items for that order 
  3. Update the stock_qty of the purchased products 
  4. If any step fails, ROLLBACK the entire transaction. Otherwise, COMMIT. 
Write the complete BEGIN...COMMIT/ROLLBACK block. 

BEGIN TRANSACTION;

INSERT INTO orders
VALUES (
1011,
102,
DATE('now'),
'Pending',
1598.00
);

INSERT INTO order_items
VALUES (
5016,
1011,
206,
1,
1299.00,
0
);

INSERT INTO order_items
VALUES (
5017,
1011,
208,
1,
599.00,
0
);

UPDATE products
SET stock_qty = stock_qty - 1
WHERE product_id = 206;

UPDATE products
SET stock_qty = stock_qty - 1
WHERE product_id = 208;

COMMIT;

If any statement fails, execute: ROLLBACK;

This transaction ensures that the order creation, order item insertion, and stock updates are treated as a single unit of work. If any step fails, all previous changes are undone using ROLLBACK, ensuring data consistency and preventing partial updates.